In [58]:
import pandas as pd
import pandas as pd
from win32com.client import Dispatch
from win32com.client import DispatchEx


In [59]:
def get_sheet(workbook, target_name):
    target = target_name.strip().lower()

    for ws in workbook.Worksheets:
        name = ws.Name.strip().lower()
        if name == target:
            return ws

    raise Exception(f"Feuille '{target_name}' introuvable")

In [60]:
def get_or_open(excel, path):
    for wb in excel.Workbooks:
        if wb.FullName.lower() == path.lower():
            return wb
    return excel.Workbooks.Open(path)

In [61]:
path_fichier_vl_n = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Old fund\Ficheir VL n.xlsx"
path_fichier_vl_n_1 = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Old fund\Fichier VL n_1.xlsx"
path_comptes_annuells = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Old fund\2025.12. Sancy  .Comptes annuels.xlsm"

# Extracting Data

## Extracting Balance, Inventaire, PVMV N

In [62]:
balance_n = pd.read_excel(path_fichier_vl_n, "Balance ", header=None, dtype=str)
inventaire_n = pd.read_excel(path_fichier_vl_n, "Inventaire", header=None, dtype=str)
pvmv = pd.read_excel(path_fichier_vl_n, "PVMV", header=None, dtype=str)

## Extracting Balance, Inventaire N-1

In [63]:
balance_n_1 = pd.read_excel(path_fichier_vl_n_1, "Balance ", header=None, dtype=str)
inventaire_n_1 = pd.read_excel(path_fichier_vl_n_1, "Inventaire", header=None, dtype=str)

# Excel automation 

## Copy paste Balance N, Inventaire N, and PVMV

# Recode Excel Macros with python

### Fonction de format Merlin

In [ ]:
def recreer_code_inventaire(workbook):
    """
    Equivalent Python de RecreerCodeInventaire()
    Remplit la colonne B de 'Inventaire N-1' avec les codes 
    de 'Base titres' (colonnes E+F+G) si colonne D != 'AU'
    """
    
    ws_inv  = get_sheet(workbook, "Inventaire N-1")
    ws_base = get_sheet(workbook, "Base titres")
    
    # Dernières lignes utilisées (colonne C)
    last_row_inv  = ws_inv.Cells(ws_inv.Rows.Count, "C").End(-4162).Row   # xlUp = -4162
    last_row_base = ws_base.Cells(ws_base.Rows.Count, "C").End(-4162).Row
    
    # Charger "Base titres" colonne C, D, E, F, G en DataFrame
    base_data = []
    for row in range(2, last_row_base + 1):
        base_data.append({
            "titre": ws_base.Cells(row, "C").Value,
            "nature": ws_base.Cells(row, "D").Value,
            "E": ws_base.Cells(row, "E").Value or "",
            "F": ws_base.Cells(row, "F").Value or "",
            "G": ws_base.Cells(row, "G").Value or "",
        })
    df_base = pd.DataFrame(base_data).set_index("titre")
    
    # Boucle sur Inventaire N-1
    for i in range(2, last_row_inv + 1):
        titre = ws_inv.Cells(i, "C").Value
        
        if titre and str(titre).strip():
            if titre in df_base.index:
                row_base = df_base.loc[titre]
                
                if row_base["nature"] != "AU":
                    code = str(row_base["E"]) + str(row_base["F"]) + str(row_base["G"])
                    ws_inv.Cells(i, "B").Value = code
    
    print("Codes ISIN créés.")

# Appel
# recreer_code_inventaire(comptes_annuells)

### Remplissage automatique

In [65]:
def convertir_colonne_a_texte(workbook):

    for nom_feuille in ["Balance", "Balance N-1"]:

        try:
            ws = workbook.Worksheets(nom_feuille)
        except Exception:
            print(f"Feuille {nom_feuille} introuvable")
            continue

        last_row = ws.Cells(ws.Rows.Count, 1).End(-4162).Row

        for row in range(1, last_row + 1):
            value = ws.Cells(row, 1).Value

            if value not in (None, ""):
                ws.Cells(row, 1).Value = "'" + str(value)

    print("Les données ont été remplies automatiquement")

### Suppression des données

In [66]:
def effacer_donnees_et_mise_en_forme(workbook):
    """
    Equivalent Python de EffacerDonneesEtMiseEnFormeOnglets()
    Efface les données et la mise en forme des feuilles listées.
    """
    
    onglets = ["Balance", "Balance N-1", "Inventaire", "Inventaire N-1", "PVMV"]
    
    for nom_feuille in onglets:
        try:
            ws = get_sheet(workbook, nom_feuille)
            ws.Cells.ClearContents()  # Effacer les données
            ws.Cells.ClearFormats()   # Effacer la mise en forme
            print(f"✅ '{nom_feuille}' effacé")
        except Exception:
            print(f"⚠️ Feuille '{nom_feuille}' introuvable, ignorée.")
    
    print("✅ Les données ont été réinitialisées")

### Fonction copier coller des données

In [67]:
def paste_df_to_sheet(workbook, sheet_name, df):
    ws = get_sheet(workbook, sheet_name)
    
    df_clean = df.where(pd.notna(df), other=" ")
    
    # Formater les colonnes datetime en string DD/MM/YYYY
    for col in df_clean.columns:
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            print("dates found")
            df_clean[col] = df_clean[col].dt.strftime("%d/%m/%Y")
    
    rows, cols = df_clean.shape
    rng = ws.Range(ws.Cells(1, 1), ws.Cells(rows, cols))
    rng.Value = df_clean.values.tolist()
    
    print(f"'{sheet_name}' collé")
    # ← plus de Save/Close ici

### Fonction pour appliquer la numérotation des pages

In [68]:
def appliquer_numerotation_continue(workbook):
    
    onglets_exclus = [
        "Balance", "Balance N-1", "PVMV", "Récap Cessions",
        "Inventaire", "Inventaire N-1", "Procédure", "Controles",
        "Fusion inventaires", "Base Créances", "Base frais", "Code pays"
    ]
    
    def get_page_count(ws):
        """Utilise ExecuteExcel4Macro comme le fait le VBA original."""
        try:
            ws.Activate()
            count = workbook.Application.ExecuteExcel4Macro("GET.DOCUMENT(50)")
            return int(count) if count and count > 0 else 0
        except:
            return 0

    # ── Étape 1 : Total des pages
    total_pages = 0
    for ws in workbook.Worksheets:
        if ws.Name not in onglets_exclus:
            page_count = get_page_count(ws)
            total_pages += page_count

    print(f"📄 Total pages : {total_pages}")

    # ── Étape 2 : Numérotation continue
    page_depart = 1
    for ws in workbook.Worksheets:
        if ws.Name not in onglets_exclus:
            page_count = get_page_count(ws)
            
            ws.PageSetup.DifferentFirstPageHeaderFooter = False
            
            if ws.Name == "Présentation":
                ws.PageSetup.RightFooter = (
                    f'&"Calibri"&9&KFFFFFF Page {page_depart} sur {total_pages}'
                )
            else:
                ws.PageSetup.RightFooter = (
                    f'&"Calibri"&9&K000000 Page {page_depart} sur {total_pages}'
                )
            
            page_depart += page_count
        else:
            ws.PageSetup.RightFooter = ""

    print("✅ Numérotation des pages appliquée avec succès !")

# Appel


### Ouverture fichier Excel

In [69]:
import pandas as pd

def open_excel_file(path) : 
    print("Ouverture du fichier")
    excel = excel = DispatchEx("Excel.Application")
    excel.Visible = False
    file_content = get_or_open(excel, path)
    print("Fichier ouvert avec succées")
    return file_content, excel

# Test automation

In [70]:
#ouverture du fichier
comptes_annuells, excel = open_excel_file(path_comptes_annuells)
#copier coller de la balance

Ouverture du fichier
Fichier ouvert avec succées


In [71]:
effacer_donnees_et_mise_en_forme(comptes_annuells)

✅ 'Balance' effacé
✅ 'Balance N-1' effacé
✅ 'Inventaire' effacé
✅ 'Inventaire N-1' effacé
✅ 'PVMV' effacé
✅ Les données ont été réinitialisées


In [72]:
paste_df_to_sheet(comptes_annuells, "Balance", balance_n)

'Balance' collé


In [73]:
paste_df_to_sheet(comptes_annuells, "Inventaire", inventaire_n)

'Inventaire' collé


In [74]:
paste_df_to_sheet(comptes_annuells, "PVMV", pvmv)

'PVMV' collé


In [75]:
paste_df_to_sheet(comptes_annuells, "Balance N-1", balance_n_1)

'Balance N-1' collé


In [76]:
paste_df_to_sheet(comptes_annuells, "Inventaire N-1", inventaire_n_1)

'Inventaire N-1' collé


In [77]:
convertir_colonne_a_texte(comptes_annuells)

Les données ont été remplies automatiquement


In [78]:
# appliquer_numerotation_continue(comptes_annuells)

📄 Total pages : 0
✅ Numérotation des pages appliquée avec succès !


In [79]:
# Fermeture propre une seule fois
comptes_annuells.Save()
comptes_annuells.Close()
excel.Quit()
print("Fichier fermé proprement")

Fichier fermé proprement


In [80]:
# inventaire_n